# 05 — Express.js & REST APIs

Express is the most popular Node.js web framework — nearly every backend Node.js interview will cover it.

---

## Table of Contents
1. HTTP Module (Built-in)
2. Express Basics
3. Middleware — The Core Concept
4. Routing
5. Request & Response Objects
6. RESTful API Design
7. Error Handling Middleware
8. Common Middleware Libraries
9. Express vs Alternatives
10. Interview Questions

---
## 1. HTTP Module (Built-in)

Before Express, understand the raw `http` module — interviewers love asking "What does Express do on top of http?"

In [ ]:
const http = require('http');

// Raw Node.js HTTP server (no Express)
const server = http.createServer((req, res) => {
    // req = IncomingMessage (Readable stream)
    // res = ServerResponse (Writable stream)

    if (req.method === 'GET' && req.url === '/') {
        res.writeHead(200, { 'Content-Type': 'application/json' });
        res.end(JSON.stringify({ message: 'Hello from raw http!' }));
    } else if (req.method === 'GET' && req.url === '/users') {
        res.writeHead(200, { 'Content-Type': 'application/json' });
        res.end(JSON.stringify([{ id: 1, name: 'Alice' }]));
    } else {
        res.writeHead(404);
        res.end('Not Found');
    }
});

// server.listen(3000);
console.log('Raw HTTP server — you handle EVERYTHING manually: routing, parsing, headers, etc.');
console.log('This is why Express exists!');

---
## 2. Express Basics

Express adds: routing, middleware pipeline, request/response helpers, templating support.

```javascript
const express = require('express');
const app = express();

// Built-in middleware for JSON body parsing
app.use(express.json());

// Route handler
app.get('/', (req, res) => {
    res.json({ message: 'Hello from Express!' });
});

app.listen(3000, () => {
    console.log('Server running on port 3000');
});
```

### What Express adds over raw `http`:
- **Routing** — `app.get()`, `app.post()`, route params, query parsing
- **Middleware** — pluggable request processing pipeline
- **Helpers** — `res.json()`, `res.send()`, `res.status()`, `req.params`, `req.query`
- **Error handling** — centralized error middleware

---
## 3. Middleware — The Core Concept

Middleware functions have access to `req`, `res`, and `next`. They execute **in order** and form a pipeline.

```
Request → [Middleware 1] → [Middleware 2] → [Route Handler] → Response
             ↓                  ↓                  ↓
          next()             next()            res.json()
```

### Types of middleware:
1. **Application-level** — `app.use()`, `app.get()`, etc.
2. **Router-level** — `router.use()`, `router.get()`, etc.
3. **Error-handling** — `(err, req, res, next)` — 4 parameters!
4. **Built-in** — `express.json()`, `express.static()`, `express.urlencoded()`
5. **Third-party** — `cors`, `helmet`, `morgan`, etc.

In [ ]:
// Middleware execution flow

// Simulating Express middleware behavior
function simulateMiddleware() {
    const middlewares = [];

    function use(fn) { middlewares.push(fn); }

    function run(req, res) {
        let idx = 0;
        function next(err) {
            if (err) return console.error('Error:', err);
            if (idx >= middlewares.length) return;
            const mw = middlewares[idx++];
            mw(req, res, next);
        }
        next();
    }

    return { use, run };
}

const app = simulateMiddleware();

// Logger middleware
app.use((req, res, next) => {
    console.log(`[LOG] ${req.method} ${req.url}`);
    req.startTime = Date.now();
    next(); // MUST call next() or request hangs!
});

// Auth middleware
app.use((req, res, next) => {
    if (req.headers.authorization) {
        console.log('[AUTH] Authorized');
        next();
    } else {
        console.log('[AUTH] Unauthorized — stopping here');
        // Don't call next() → request stops
    }
});

// Route handler
app.use((req, res, next) => {
    console.log(`[HANDLER] Processing request (took ${Date.now() - req.startTime}ms)`);
});

console.log('--- With auth header ---');
app.run({ method: 'GET', url: '/api/users', headers: { authorization: 'Bearer xyz' } }, {});

console.log('\n--- Without auth header ---');
app.run({ method: 'GET', url: '/api/users', headers: {} }, {});

---
## 4. Routing

```javascript
const express = require('express');
const app = express();

// Basic routes
app.get('/users', getUsers);       // GET    /users
app.get('/users/:id', getUser);    // GET    /users/123
app.post('/users', createUser);    // POST   /users
app.put('/users/:id', updateUser); // PUT    /users/123
app.patch('/users/:id', patchUser);// PATCH  /users/123
app.delete('/users/:id', delUser); // DELETE /users/123

// Route parameters
app.get('/users/:userId/posts/:postId', (req, res) => {
    // req.params = { userId: '123', postId: '456' }
});

// Query strings: GET /search?q=node&page=2
app.get('/search', (req, res) => {
    // req.query = { q: 'node', page: '2' }
});

// Router (for modular routing)
const router = express.Router();
router.get('/', getUsers);
router.get('/:id', getUser);
router.post('/', createUser);
app.use('/api/users', router); // Mount at /api/users
```

### Route parameter patterns:
```javascript
// Optional parameter
app.get('/users/:id?', handler);

// Regex constraint
app.get('/users/:id(\\d+)', handler); // only numeric IDs

// Wildcard
app.get('/files/*', handler); // matches /files/any/path/here
```

---
## 5. Request & Response Objects

### Request (`req`) — key properties:
```javascript
req.params    // Route parameters: /users/:id → { id: '123' }
req.query     // Query string: ?page=2 → { page: '2' }
req.body      // Parsed body (needs express.json() middleware)
req.headers   // Request headers
req.method    // GET, POST, PUT, DELETE, etc.
req.url       // Request URL
req.path      // Path portion of URL
req.ip        // Client IP address
req.cookies   // Cookies (needs cookie-parser middleware)
req.get('header-name') // Get specific header
```

### Response (`res`) — key methods:
```javascript
res.status(200)                    // Set status code
res.json({ data: 'value' })       // Send JSON (sets Content-Type automatically)
res.send('text')                   // Send string response
res.sendFile('/path/to/file')      // Send a file
res.redirect('/new-url')           // Redirect (302 by default)
res.redirect(301, '/new-url')      // Permanent redirect
res.set('Header', 'Value')         // Set response header
res.cookie('name', 'value')        // Set cookie
res.status(404).json({ error: 'Not found' }) // Chain status + response
```

---
## 6. RESTful API Design

REST (Representational State Transfer) principles:

### HTTP Methods & CRUD:
| Method | Action | Route | Body | Idempotent? |
|--------|--------|-------|------|------------|
| GET | Read (list) | `/users` | No | Yes |
| GET | Read (one) | `/users/:id` | No | Yes |
| POST | Create | `/users` | Yes | No |
| PUT | Full update | `/users/:id` | Yes | Yes |
| PATCH | Partial update | `/users/:id` | Yes | Yes |
| DELETE | Delete | `/users/:id` | No | Yes |

### Status Codes to Memorize:
```
200 OK            — Successful GET/PUT/PATCH
201 Created       — Successful POST (resource created)
204 No Content    — Successful DELETE
400 Bad Request   — Invalid input / validation error
401 Unauthorized  — No auth / invalid credentials
403 Forbidden     — Authenticated but not authorized
404 Not Found     — Resource doesn't exist
409 Conflict      — Duplicate resource
422 Unprocessable — Valid syntax but semantic errors
429 Too Many Req  — Rate limited
500 Internal Error— Server bug
503 Unavailable   — Server overloaded / maintenance
```

### Good REST Practices:
- Use **nouns** for resources: `/users`, NOT `/getUsers`
- Use **plural** names: `/users`, NOT `/user`
- Nest related resources: `/users/:id/posts`
- Use query params for filtering/sorting: `/users?role=admin&sort=name`
- Version your API: `/api/v1/users`
- Always return consistent error format

In [ ]:
// Complete REST API example structure

const apiDesign = {
    'GET /api/v1/users':          'List all users (with pagination)',
    'GET /api/v1/users/:id':      'Get single user',
    'POST /api/v1/users':         'Create user → returns 201',
    'PUT /api/v1/users/:id':      'Full update user',
    'PATCH /api/v1/users/:id':    'Partial update user',
    'DELETE /api/v1/users/:id':   'Delete user → returns 204',
    'GET /api/v1/users/:id/posts':'Get user\'s posts (nested resource)',
};

// Consistent error response format
const errorResponse = {
    status: 'error',
    statusCode: 404,
    message: 'User not found',
    timestamp: new Date().toISOString()
};

// Consistent success response format
const successResponse = {
    status: 'success',
    data: { id: 1, name: 'Alice', email: 'alice@example.com' },
    meta: { page: 1, limit: 20, total: 100 } // for list endpoints
};

console.log('API Design:', JSON.stringify(apiDesign, null, 2));
console.log('\nError format:', JSON.stringify(errorResponse, null, 2));
console.log('\nSuccess format:', JSON.stringify(successResponse, null, 2));

---
## 7. Error Handling Middleware

Express error-handling middleware has **4 parameters** — `(err, req, res, next)`.

```javascript
// Custom error class
class AppError extends Error {
    constructor(message, statusCode) {
        super(message);
        this.statusCode = statusCode;
        this.isOperational = true;
    }
}

// In route handlers — throw or pass to next()
app.get('/users/:id', async (req, res, next) => {
    try {
        const user = await User.findById(req.params.id);
        if (!user) throw new AppError('User not found', 404);
        res.json(user);
    } catch (err) {
        next(err); // forwards to error middleware
    }
});

// Centralized error handler (must be LAST middleware)
app.use((err, req, res, next) => {
    const statusCode = err.statusCode || 500;
    res.status(statusCode).json({
        status: 'error',
        statusCode,
        message: err.isOperational ? err.message : 'Internal server error',
    });
});
```

### Async error wrapper (DRY pattern):
```javascript
const asyncHandler = (fn) => (req, res, next) => {
    Promise.resolve(fn(req, res, next)).catch(next);
};

// Now you don't need try/catch in every route
app.get('/users/:id', asyncHandler(async (req, res) => {
    const user = await User.findById(req.params.id);
    if (!user) throw new AppError('User not found', 404);
    res.json(user);
}));
```

---
## 8. Common Middleware Libraries

| Middleware | Purpose |
|-----------|--------|
| `express.json()` | Parse JSON request bodies |
| `express.urlencoded()` | Parse URL-encoded form data |
| `express.static()` | Serve static files (HTML, CSS, images) |
| `cors` | Enable Cross-Origin Resource Sharing |
| `helmet` | Security headers (XSS, clickjacking protection) |
| `morgan` | HTTP request logging |
| `compression` | Gzip response compression |
| `express-rate-limit` | Rate limiting |
| `cookie-parser` | Parse cookies from requests |
| `express-validator` | Input validation |
| `multer` | File upload handling |
| `passport` | Authentication strategies |

---
## 9. Express vs Alternatives

| Framework | Type | Key Feature | Performance |
|-----------|------|------------|------------|
| **Express** | Minimal | Middleware-based, huge ecosystem | Moderate |
| **Fastify** | Performance | Schema-based validation, fast JSON | 2-3x Express |
| **Koa** | Modern | async/await native, no bundled middleware | Similar |
| **NestJS** | Full framework | TypeScript, Angular-style DI, decorators | On top of Express/Fastify |
| **Hapi** | Enterprise | Configuration-driven, built-in validation | Similar |

> **Interview Tip:** If asked "Why Express?" — it's the industry standard with the largest ecosystem. If asked about alternatives, mention Fastify for performance or NestJS for TypeScript enterprise apps.

---
## 10. Interview Questions & Answers

### Q1: What is middleware in Express?
**A:** Functions that have access to req, res, and next. They execute sequentially, can modify the request/response, end the request cycle, or call next() to pass control to the next middleware. Express is essentially a series of middleware calls.

### Q2: What's the difference between `app.use()` and `app.get()`?
**A:** `app.use()` matches ALL HTTP methods and partial paths (prefix matching). `app.get()` only matches GET requests and exact paths. `app.use('/api')` matches `/api`, `/api/users`, `/api/anything`. `app.get('/api')` only matches `/api`.

### Q3: How does Express error handling work?
**A:** Error-handling middleware has 4 parameters: `(err, req, res, next)`. When you call `next(error)` in any middleware, Express skips all remaining non-error middleware and goes to the error handler. The error middleware should be the last `app.use()` call.

### Q4: What's the difference between PUT and PATCH?
**A:** PUT replaces the entire resource (you must send all fields). PATCH applies a partial update (only send the fields you want to change). Both are idempotent — calling them multiple times produces the same result.

### Q5: How do you handle async errors in Express?
**A:** Either use try/catch with `next(err)` in every route, or create an `asyncHandler` wrapper that catches rejected Promises and forwards them to `next()`. Express 5 will handle async errors automatically.

### Q6: What is CORS and how do you handle it?
**A:** CORS (Cross-Origin Resource Sharing) is a browser security mechanism that blocks requests from different origins. In Express, use the `cors` middleware package. Configure it to allow specific origins, methods, and headers for your API.